In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "GA"
df = pd.read_excel(file_path, sheet_name=sheet_name)

# ========= 2) 参数模板 =========
params = {
    "dim": 2,
    "lb": np.array([-10.0, -10.0]),
    "ub": np.array([10.0, 10.0]),
    "pop_size": 40,
    "n_gen": 100,
    "crossover_rate": 0.8,
    "mutation_rate": 0.1
}

def obj(x): return np.sum((x-3)**2)

pop = params["lb"] + (params["ub"]-params["lb"]) * np.random.rand(params["pop_size"], params["dim"])

def select(pop, fit, k=3):
    idx = np.random.choice(len(pop), k, replace=False)
    return pop[idx[np.argmin(fit[idx])]]

for _ in range(params["n_gen"]):
    fit = np.array([obj(ind) for ind in pop])
    new_pop = []
    while len(new_pop) < params["pop_size"]:
        p1, p2 = select(pop, fit), select(pop, fit)
        if np.random.rand() < params["crossover_rate"]:
            a = np.random.rand(params["dim"])
            c1, c2 = a*p1 + (1-a)*p2, a*p2 + (1-a)*p1
        else:
            c1, c2 = p1.copy(), p2.copy()

        for c in [c1, c2]:
            m = np.random.rand(params["dim"]) < params["mutation_rate"]
            c[m] += 0.1*np.random.randn(np.sum(m))
            c[:] = np.clip(c, params["lb"], params["ub"])
            new_pop.append(c)
            if len(new_pop) >= params["pop_size"]:
                break
    pop = np.array(new_pop)

fit = np.array([obj(ind) for ind in pop])
best = pop[np.argmin(fit)]
print(best, np.min(fit))


In [ ]:
"""
遗传算法

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "遗传算法.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
DIMENSION = 2  # TODO: 请填写[变量维度]，说明：决策变量个数。
LOWER_BOUND = -5  # TODO: 请填写[变量下界]，说明：可为数值或自行扩展为数组。
UPPER_BOUND = 5  # TODO: 请填写[变量上界]，说明：必须大于下界。
POP_SIZE = 50  # TODO: 请填写[种群规模]，说明：越大搜索越充分但更慢。
GENERATIONS = 100  # TODO: 请填写[迭代代数]，说明：正整数。
MUTATION_SCALE = 0.1  # TODO: 请填写[变异强度]，说明：过大不稳定，过小易早熟。
TARGET_VECTOR = np.array([1.0, 2.0])  # TODO: 请填写[示例目标向量]，说明：请替换为真实适应度函数参数。



REQUIRES_DATA = False  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def fitness(x):
    # TODO: 请填写[适应度函数]，说明：遗传算法默认最大化该值。
    return -np.sum((x - TARGET_VECTOR) ** 2)


def run_model(data: pd.DataFrame) -> None:
    rng = np.random.default_rng(RANDOM_STATE)
    pop = rng.uniform(LOWER_BOUND, UPPER_BOUND, size=(POP_SIZE, DIMENSION))
    for _ in range(GENERATIONS):
        scores = np.array([fitness(ind) for ind in pop])
        parents = pop[np.argsort(scores)[-POP_SIZE // 2:]]
        children = []
        while len(children) < POP_SIZE:
            a, b = parents[rng.integers(len(parents), size=2)]
            child = (a + b) / 2 + rng.normal(0, MUTATION_SCALE, DIMENSION)
            children.append(np.clip(child, LOWER_BOUND, UPPER_BOUND))
        pop = np.array(children)
    best = pop[np.argmax([fitness(ind) for ind in pop])]
    print("最优个体:", best, "适应度:", fitness(best))


if __name__ == "__main__":
    df = load_data()
    run_model(df)


# 遗传算法

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

遗传算法 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

离散、非凸、黑箱目标函数优化。

## 局限性

计算量较大，参数设置影响明显。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。
